In [34]:
# библиотеки
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import scipy.stats as sts
from itertools import combinations

In [5]:
# load data
%store -r data

In [20]:
# Выделяем количественные переменные
quantitative_data = data[['score', 'scored_by', 'volumes', 'chapters']]

# Строим корреляционную матрицу
correlation_matrix = quantitative_data.corr().round(3)
print(correlation_matrix)

           score  scored_by  volumes  chapters
score      1.000      0.305    0.365     0.221
scored_by  0.305      1.000    0.308     0.204
volumes    0.365      0.308    1.000     0.704
chapters   0.221      0.204    0.704     1.000


1. Связь результативного признака (score) с факторами:

score и scored_by (r = 0.305)
Умеренная положительная связь.
→ Чем больше пользователей оценило аниме, тем немного выше его рейтинг.

score и volumes (r = 0.365)
Умеренная положительная связь.
→ Аниме с большим количеством томов имеют тенденцию к более высоким оценкам.

score и chapters (r = 0.221)
Слабая положительная связь.
→ Количество глав почти не влияет на рейтинг.

2. Межфакторная корреляция:

scored_by и volumes (r = 0.308)
Умеренная положительная связь.
→ Многотомные аниме чаще оцениваются пользователями.

scored_by и chapters (r = 0.204)
Слабая положительная связь.
→ Почти нет зависимости между количеством глав и популярностью.

volumes и chapters (r = 0.704)
Сильная положительная связь.
→ Ожидаемо: чем больше томов, тем больше глав. Проблема мультиколлинеарности!

In [40]:
variables = ['score', 'scored_by', 'volumes', 'chapters']
for (var1, var2) in combinations(variables, 2):
    r, p = sts.pearsonr(data[var1], data[var2])
    print(f"{var1} & {var2}: r = {r:.3f}, p-value = {p:.3f}")

score & scored_by: r = 0.305, p-value = 0.000
score & volumes: r = 0.365, p-value = 0.000
score & chapters: r = 0.221, p-value = 0.000
scored_by & volumes: r = 0.308, p-value = 0.000
scored_by & chapters: r = 0.204, p-value = 0.000
volumes & chapters: r = 0.704, p-value = 0.000


In [41]:
# Преобразуем статус в бинарную переменную
data['status_binary'] = data['status'].apply(lambda x: 1 if x == 'finished' else 0)
print(data['status_binary'].value_counts())

if len(data['status_binary'].unique()) > 1:
    r_pb, p_value = sts.pointbiserialr(data['status_binary'], data['score'])
    print(f"Коэффициент корреляции: {r_pb:.3f}, p-value: {p_value:.3f}")
else:
    print("Ошибка: статус не варьируется (все значения одинаковы).")

status_binary
1    15637
0      109
Name: count, dtype: int64
Коэффициент корреляции: -0.048, p-value: 0.000


In [47]:
# Создадим бинарную переменную для жанра "Romance"
data['is_romance'] = data['genres'].apply(lambda x: 1 if 'Romance' in x else 0)
print(data['is_romance'].value_counts())

# Проверка корреляции
if len(data['is_romance'].unique()) > 1:
    r_pb, p_value = sts.pointbiserialr(data['is_romance'], data['score'])
    print(f"Коэффициент корреляции: {r_pb:.3f}, p-value: {p_value:.3f}")
else:
    print("Ошибка: статус не варьируется (все значения одинаковы).")

is_romance
0    10681
1     5065
Name: count, dtype: int64
Коэффициент корреляции: 0.061, p-value: 0.000
